In [53]:
from datasets import load_dataset
import os
import pandas as pd

Para cada linha:
1- Transformar label em df
2- Calcular a media de todos os especialistas para cada emoção
3- Pegar apenas as 27 emoções
4- Discretizar a label por um limiar (binarizar, 0 ou 1 para facilitar) - pode ser um parâmetro de teste
5- Ai o modelo vai prever quais emoções/expressoes da lista que tem, e vamos por 1 nas que foram citadas e 0 nas outras - assim vamos ter um limiar.

In [54]:
ds = load_dataset("laion/emonet-face-hq", split="train")


In [75]:
ds[0]

{'path': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1024x1024>,
 'prompt': 'an authentic, realistic closeup image of a South Asian non-binary man of 70 years who seems to experience spite, sadism, malevolence, malice, desire to harm, schadenfreude. Strong facial expression of spite, sadism, malevolence, malice, desire to harm, schadenfreude.',
 'age': 6,
 'ethnicity': 8,
 'gender': 3,
 'emotion': 'spite, sadism, malevolence, malice, desire to harm, schadenfreude',
 'subset': 1,
 'label': "[{'human-1': {'Cognitive States and Processes|Concentration': 3, 'Cognitive States and Processes|Confusion': 0, 'Longing & Lust|Infatuation': 0, 'Longing & Lust|Longing': 3, 'Longing & Lust|Sexual Lust': 0, 'Negative High-Energy Emotions|Anger': 1, 'Negative High-Energy Emotions|Disgust': 1, 'Negative High-Energy Emotions|Distress': 1, 'Negative High-Energy Emotions|Fear': 0, 'Negative High-Energy Emotions|Impatience and Irritability': 1, 'Negative High-Energy Emotions|Malevolence/Malice': 0

In [56]:
import json
import ast
import pandas as pd

def parse_label_field(x):
    """
    Ensures that the 'label' field becomes a proper Python object (list of dicts),
    regardless of whether it is JSON, a Python-like string, or already a Python object.
    """
    if isinstance(x, list):
        return x  # already a decoded Python object

    if isinstance(x, str):
        x = x.strip()

        # Try parsing as valid JSON
        try:
            return json.loads(x)
        except:
            pass

        # Try parsing as Python literal (supports single quotes)
        try:
            return ast.literal_eval(x)
        except:
            pass

        raise ValueError(f"Invalid label format: {x[:100]}")

    raise TypeError(f"Invalid type ({type(x)}). Expected str or list.")


def clean_emotion_name(full_name):
    """
    Extracts only the part after the '|' character.
    Example: 'Cognitive States|Concentration' -> 'Concentration'
    """
    if "|" in full_name:
        return full_name.split("|", 1)[1].strip()
    return full_name.strip()


def compute_mean_scores(label_field):
    """
    Processes the 'label' field (string or list) and returns a dictionary
    with mean emotion scores, with cleaned emotion names.
    """
    data = parse_label_field(label_field)

    # Extract dicts from each annotator (human-1, human-2, ...)
    human_dicts = [list(h.values())[0] for h in data]

    emotions = human_dicts[0].keys()
    mean_scores = {}

    for emo in emotions:
        cleaned_name = clean_emotion_name(emo)
        vals = [h[emo] for h in human_dicts]
        mean_scores[cleaned_name] = sum(vals) / len(vals)

    return mean_scores


# Convert HF dataset to pandas
df = ds.to_pandas()

# Compute mean emotion scores
mean_scores_list = df["label"].apply(compute_mean_scores)

# Convert dict into dataframe columns
mean_scores_df = pd.DataFrame(mean_scores_list.tolist())

# Merge with original dataframe
df_final = pd.concat([df, mean_scores_df], axis=1)

df_final.head()


,path,prompt,age,ethnicity,gender,emotion,subset,label,Concentration,Confusion,...,Interest,Pleasure/Ecstasy,Teasing,Triumph,Affection,Contemplation,Contentment,Pride,Relief,Thankfulness/Gratitude
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Sou...",6,8,3,"spite, sadism, malevolence, malice, desire to ...",1,[{'human-1': {'Cognitive States and Processes|...,2.75,0.0,...,1.0,0.00,0.00,0.50,0.25,0.25,0.00,0.75,0.0,0.75
1,{'bytes': b'RIFFbR\x12\x00WEBPVP8LUR\x12\x00/\...,Very realistic high quality portrait DSLR phot...,3,9,0,genuine silliness and jesting,1,[{'human-1': {'Cognitive States and Processes|...,2.25,0.5,...,0.5,0.00,0.25,0.00,1.75,0.75,0.50,0.25,0.0,1.00
2,{'bytes': b'RIFF:\xa1\x0e\x00WEBPVP8L.\xa1\x0e...,Very realistic high quality portrait DSLR phot...,0,11,4,genuine subtle laughter and silliness,1,[{'human-1': {'Cognitive States and Processes|...,3.75,0.0,...,2.5,0.75,0.50,0.75,2.50,3.00,2.25,3.00,1.0,0.50
3,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Nor...",7,7,4,"jealousy, envy, covetousness",1,[{'human-1': {'Cognitive States and Processes|...,2.75,0.5,...,0.5,0.00,0.00,0.00,0.50,1.00,0.75,0.00,0.0,1.00
4,{'bytes': b'RIFF~\xc2\x10\x00WEBPVP8Lr\xc2\x10...,Very realistic high quality portrait DSLR phot...,7,3,0,genuine subtle playfulness and joviality,1,[{'human-1': {'Cognitive States and Processes|...,1.25,0.0,...,1.0,0.00,0.00,0.00,0.25,1.25,1.25,0.25,0.0,0.25


In [57]:
len(df_final.columns)

48

In [58]:
df_final.iloc[22]

path                                            {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...
prompt                                          an authentic, realistic closeup image of a Ame...
age                                                                                             2
ethnicity                                                                                       0
gender                                                                                          0
emotion                                              ecstasy, pleasure, bliss, rapture, Beatitude
subset                                                                                          1
label                                           [{'human-1': {'Cognitive States and Processes|...
Concentration                                                                                2.75
Confusion                                                                                     0.0
Infatuation         

In [ ]:
# Identify which columns are emotion columns.
# Adjust this list if you have additional non-emotion columns
emotion_cols = [
    col for col in df_final.columns
    if col not in ["path", "prompt", "age", "ethnicity", "gender", "emotion", "subset", "label"]
]

def binarize_row_overwrite(row):
    """
    Overwrites emotion columns with binary values:
    1 for the emotion(s) with the maximum score and 0 for the rest.
    If multiple emotions share the maximum score, all are set to 1.
    """
    # Extract only emotion scores
    scores = row[emotion_cols]

    # Find maximum value for this row
    max_value = scores.max()

    # Create binary vector
    binary = (scores == max_value).astype(int)

    # Replace original values with binary ones
    row[emotion_cols] = binary

    return row


# Apply to entire dataframe
df_final = df_final.apply(binarize_row_overwrite, axis=1)

df_final.head()


,path,prompt,age,ethnicity,gender,emotion,subset,label,Concentration,Confusion,...,Interest,Pleasure/Ecstasy,Teasing,Triumph,Affection,Contemplation,Contentment,Pride,Relief,Thankfulness/Gratitude
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Sou...",6,8,3,"spite, sadism, malevolence, malice, desire to ...",1,[{'human-1': {'Cognitive States and Processes|...,1,0,...,0,0,0,0,0,0,0,0,0,0
1,{'bytes': b'RIFFbR\x12\x00WEBPVP8LUR\x12\x00/\...,Very realistic high quality portrait DSLR phot...,3,9,0,genuine silliness and jesting,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
2,{'bytes': b'RIFF:\xa1\x0e\x00WEBPVP8L.\xa1\x0e...,Very realistic high quality portrait DSLR phot...,0,11,4,genuine subtle laughter and silliness,1,[{'human-1': {'Cognitive States and Processes|...,1,0,...,0,0,0,0,0,0,0,0,0,0
3,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Nor...",7,7,4,"jealousy, envy, covetousness",1,[{'human-1': {'Cognitive States and Processes|...,1,0,...,0,0,0,0,0,0,0,0,0,0
4,{'bytes': b'RIFF~\xc2\x10\x00WEBPVP8Lr\xc2\x10...,Very realistic high quality portrait DSLR phot...,7,3,0,genuine subtle playfulness and joviality,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
df_final.columns

Index(['path', 'prompt', 'age', 'ethnicity', 'gender', 'emotion', 'subset',
       'label', 'Concentration', 'Confusion', 'Infatuation', 'Longing',
       'Sexual Lust', 'Anger', 'Disgust', 'Distress', 'Fear',
       'Impatience and Irritability', 'Malevolence/Malice', 'Bitterness',
       'Contempt', 'Disappointment', 'Doubt', 'Embarrassment',
       'Emotional Numbness', 'Helplessness', 'Jealousy & Envy', 'Sadness',
       'Shame', 'Fatigue/Exhaustion',
       'Intoxication/Altered States of Consciousness', 'Pain', 'Sourness',
       'Amusement', 'Astonishment/Surprise', 'Awe', 'Elation',
       'Hope/Enthusiasm/Optimism', 'Interest', 'Pleasure/Ecstasy', 'Teasing',
       'Triumph', 'Affection', 'Contemplation', 'Contentment', 'Pride',
       'Relief', 'Thankfulness/Gratitude'],
      dtype='object')

In [61]:
emotion_columns = [
    'path', 
    'prompt', 
    'age', 
    'ethnicity', 
    'gender', 
    'emotion', 
    'subset',
    'label',
    "Amusement",
    "Anger",
    "Awe",
    "Concentration",  # <-- NOT FOUND in dataset (não existe!)
    "Confusion",
    "Contemplation",
    "Contempt",
    "Contentment",
    "Longing",                   # Desire
    "Disappointment",
    "Disgust",
    "Distress",
    "Doubt",
    "Pleasure/Ecstasy",          # Ecstasy
    "Elation",
    "Embarrassment",
    "Fear",
    "Interest",
    "Infatuation",               # Love
    "Pain",
    "Pride",
    "Relief",
    "Sadness",
    "Shame",
    "Astonishment/Surprise",     # Surprise
    "Affection",                 # Sympathy
    "Triumph"
]

df_emo = df_final[emotion_columns]

In [62]:
df_emo.columns

Index(['path', 'prompt', 'age', 'ethnicity', 'gender', 'emotion', 'subset',
       'label', 'Amusement', 'Anger', 'Awe', 'Concentration', 'Confusion',
       'Contemplation', 'Contempt', 'Contentment', 'Longing', 'Disappointment',
       'Disgust', 'Distress', 'Doubt', 'Pleasure/Ecstasy', 'Elation',
       'Embarrassment', 'Fear', 'Interest', 'Infatuation', 'Pain', 'Pride',
       'Relief', 'Sadness', 'Shame', 'Astonishment/Surprise', 'Affection',
       'Triumph'],
      dtype='object')

In [63]:
only_emo = ['Amusement', 'Anger', 'Awe', 'Concentration', 'Confusion',
       'Contemplation', 'Contempt', 'Contentment', 'Longing', 'Disappointment',
       'Disgust', 'Distress', 'Doubt', 'Pleasure/Ecstasy', 'Elation',
       'Embarrassment', 'Fear', 'Interest', 'Infatuation', 'Pain', 'Pride',
       'Relief', 'Sadness', 'Shame', 'Astonishment/Surprise', 'Affection',
       'Triumph']

In [68]:
df_emo_final = df_emo[df_emo[only_emo].sum(axis=1) == 1]

In [ ]:
len(df_emo_final)

1883

In [73]:
df_emo_final.head(3)

,path,prompt,age,ethnicity,gender,emotion,subset,label,Amusement,Anger,...,Interest,Infatuation,Pain,Pride,Relief,Sadness,Shame,Astonishment/Surprise,Affection,Triumph
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Sou...",6,8,3,"spite, sadism, malevolence, malice, desire to ...",1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0
1,{'bytes': b'RIFFbR\x12\x00WEBPVP8LUR\x12\x00/\...,Very realistic high quality portrait DSLR phot...,3,9,0,genuine silliness and jesting,1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,1,0,0,0,0
3,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,"an authentic, realistic closeup image of a Nor...",7,7,4,"jealousy, envy, covetousness",1,[{'human-1': {'Cognitive States and Processes|...,0,0,...,0,0,0,0,0,0,0,0,0,0


In [74]:
(df_emo[only_emo] == 1).sum()

Amusement                 74
Anger                     90
Awe                       15
Concentration            804
Confusion                 45
Contemplation            223
Contempt                  30
Contentment              164
Longing                   28
Disappointment            71
Disgust                    5
Distress                  72
Doubt                     74
Pleasure/Ecstasy           6
Elation                   89
Embarrassment              3
Fear                      24
Interest                  31
Infatuation              119
Pain                       8
Pride                     74
Relief                    12
Sadness                  148
Shame                      3
Astonishment/Surprise     77
Affection                212
Triumph                    4
dtype: int64